# House Prices | minimum code 0.15755

**Introduction**

- I decided to try to solve the problem in a minimum of lines of code
- There were long hours of analysis and checks behind the scenes

![](https://rukminim1.flixcart.com/image/416/416/block-construction/y/k/v/lego-creator-hillside-house-original-imadbjrg7cgehbqy.jpeg?q=70)

- `numpy` array library
- `pandas` for data processing and analysis
- `seaborn, matplotlib` dependency plotting library
- `warnings` library for disabling messages
- `StandardScaler` standardize features by removing the mean and scaling to unit variance
- `ElasticNet` Linear regression with combined L1 and L2 priors as regularizer
- `RandomForestRegressor` a random forest regressor
- `XGBRegressor` Extreme Gradient Boosting

In [196]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

- `train` training data set
- `test` test data set
- `house_id` house Id

In [197]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')
house_id = test['Id']

- plot rectangular data as a color-encoded matrix
- remove the comments if you want to build it

In [198]:
# plt.figure(figsize=(10,8))
# sns.heatmap(train.corr(), cmap="RdBu")
# plt.show()

- `a` a list of all features that have a correlation greater than 0.5 or less than -0.5
- `b` these features will also be needed
- `train` the training set will consist of all selected features
- `test` the test set will consist of all selected features (except for the price)

In [199]:
a = list(train.corr()['SalePrice'][(train.corr()['SalePrice'] > 0.50) | (train.corr()['SalePrice'] < -0.50)].index)
b = ["MSZoning", "Utilities","BldgType","Heating","KitchenQual","SaleCondition","LandSlope"]

train = train[a + b]
a.remove('SalePrice')
test = test[a + b]

- there are gaps in the test data
- they are not in the training data

In [200]:
c = pd.DataFrame(test.isna().sum()).T
c

,OverallQual,YearBuilt,YearRemodAdd,TotalBsmtSF,1stFlrSF,GrLivArea,FullBath,TotRmsAbvGrd,GarageCars,GarageArea,MSZoning,Utilities,BldgType,Heating,KitchenQual,SaleCondition,LandSlope
0,0,0,0,1,0,0,0,0,1,1,4,2,0,0,1,0,0


- replace gaps with mean values or blank values

In [201]:
test['TotalBsmtSF'] = test['TotalBsmtSF'].fillna(test['TotalBsmtSF'].mean())
test['GarageCars'] = test['GarageCars'].fillna(test['GarageCars'].mean())
test['GarageArea'] = test['GarageArea'].fillna(test['GarageArea'].mean())
test['MSZoning'] = test['MSZoning'].fillna('')
test['Utilities'] = test['Utilities'].fillna('')
test['KitchenQual'] = test['KitchenQual'].fillna('')

- passes are gone

In [202]:
c = pd.DataFrame(test.isna().sum()).T
c

,OverallQual,YearBuilt,YearRemodAdd,TotalBsmtSF,1stFlrSF,GrLivArea,FullBath,TotRmsAbvGrd,GarageCars,GarageArea,MSZoning,Utilities,BldgType,Heating,KitchenQual,SaleCondition,LandSlope
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


- `X` dataset for training the model (all features, except for the price of the house)
- `y` label for training (house price)
- `get_dummies` Convert categorical variable into dummy/indicator variables
- `scaler` compress the numeric data so that it is between -1 and +1

In [203]:
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

X = pd.get_dummies(X, columns=b)
test = pd.get_dummies(test, columns=b)

scaler = StandardScaler()
X[a] = scaler.fit_transform(X[a])
test[a] = scaler.fit_transform(test[a])

- `result` empty dataframe
- we will create and train three models to collect three options for the price of a house
- models gave a different price, the answer lies in the middle

In [204]:
result = pd.DataFrame()

rf = RandomForestRegressor(n_estimators=100)
rf.fit(X, y)
result['rf'] = rf.predict(test)

net = ElasticNet()
net.fit(X, y)
result['net'] = net.predict(test)

xgb = XGBRegressor(n_estimators=1000, learning_rate=0.01)
xgb.fit(X, y)
result['xgb'] = xgb.predict(test)

# result

- `output` final dataset with the answer
- you can replace `maen` → `median`

In [205]:
output = pd.DataFrame({ 'Id': house_id, 'SalePrice': result.mean(axis=1) })
output.to_csv('sub.csv', index=False)

**My result was not very good (0.15755), you can add more models and then your result will be more accurate. I will be glad to any comments and suggestions. Thank you all for your attention.**